In [ ]:
!pip install selenium pandas openpyxl beautifulsoup4


In [ ]:
!pip install playwright
!playwright install


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup
import re
import pandas as pd
from playwright.sync_api import sync_playwright

Código simples inicial

In [2]:
options = Options()
# Não usar headless para garantir tudo carregado
# options.add_argument("--headless") 

driver = webdriver.Chrome(options=options)
url = "https://a.co/d/f2wXCXp"
driver.get(url)

time.sleep(7)  # espera carregar tudo

# Pega o HTML completo carregado
html = driver.page_source

# Salva localmente
with open("pagina_amazon_completa.html", "w", encoding="utf-8") as f:
    f.write(html)

driver.quit()

# Agora extrai com BeautifulSoup do HTML salvo
with open("pagina_amazon_completa.html", "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

# Extrair dados com seletores ajustados
nome = soup.select_one("#productTitle")
nome = nome.get_text(strip=True) if nome else "Nome não encontrado"

preco_inteiro = soup.select_one(".a-price-whole")
preco_fracao = soup.select_one(".a-price-fraction")
if preco_inteiro and preco_fracao:
    preco = f"R$ {preco_inteiro.get_text(strip=True)},{preco_fracao.get_text(strip=True)}"
else:
    preco = "Preço não encontrado"

avaliacoes = None
seletores_avaliacoes = [
    "#acrCustomerReviewText",
    "span[data-hook='total-review-count']",
    "#reviews-medley-footer span[data-hook='total-review-count']"
]

for seletor in seletores_avaliacoes:
    elemento = soup.select_one(seletor)
    if elemento:
        texto = elemento.get_text(strip=True)
        numeros = re.findall(r"[\d\.]+", texto)
        avaliacoes = numeros[0] if numeros else texto
        break

if not avaliacoes:
    avaliacoes = "Avaliações não encontradas"

estrelas = soup.select_one("span.a-icon-alt")
estrelas = estrelas.get_text(strip=True) if estrelas else "Nota não encontrada"

descricao = soup.select_one("#feature-bullets ul li span")
descricao = descricao.get_text(strip=True) if descricao else "Descrição não encontrada"

print(f"Nome: {nome}")
print(f"Preço: {preco}")
print(f"Avaliações: {avaliacoes}")
print(f"Estrelas: {estrelas}")
print(f"Descrição: {descricao}")

Nome: WAP Aspirador de Pó Robô ROBOT W90 3 em 1, Automático, 250ml, Sistema Antiqueda e Rodas Emborrachadas, 30W 3,6VDC Bivolt
Preço: R$ 376,,48
Avaliações: 11.934
Estrelas: 4,3 de 5 estrelas
Descrição: Conta com funções de aspirar, varrer e passar pano de forma totalmente automática e inteligente.


Código com selenium, requer login

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import re
import pandas as pd

# Parte 1: Obter detalhes do produto
options = Options()
# options.add_argument("--headless")
driver = webdriver.Chrome(options=options)
url = "https://a.co/d/f2wXCXp"
driver.get(url)
time.sleep(20)

html = driver.page_source
with open("pagina_amazon_completa.html", "w", encoding="utf-8") as f:
    f.write(html)
driver.quit()

with open("pagina_amazon_completa.html", "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

nome = soup.select_one("#productTitle")
nome = nome.get_text(strip=True) if nome else "Nome não encontrado"

preco_inteiro = soup.select_one(".a-price-whole")
preco_fracao = soup.select_one(".a-price-fraction")
if preco_inteiro and preco_fracao:
    preco = f"R$ {preco_inteiro.get_text(strip=True)},{preco_fracao.get_text(strip=True)}"
else:
    preco = "Preço não encontrado"

avaliacoes = None
seletores_avaliacoes = [
    "#acrCustomerReviewText",
    "span[data-hook='total-review-count']",
    "#reviews-medley-footer span[data-hook='total-review-count']"
]

for seletor in seletores_avaliacoes:
    elemento = soup.select_one(seletor)
    if elemento:
        texto = elemento.get_text(strip=True)
        numeros = re.findall(r"[\d\.]+", texto)
        avaliacoes = numeros[0] if numeros else texto
        break

if not avaliacoes:
    avaliacoes = "Avaliações não encontradas"

estrelas = soup.select_one("span.a-icon-alt")
estrelas = estrelas.get_text(strip=True) if estrelas else "Nota não encontrada"

descricao = soup.select_one("#feature-bullets ul li span")
descricao = descricao.get_text(strip=True) if descricao else "Descrição não encontrada"

print(f"Nome: {nome}")
print(f"Preço: {preco}")
print(f"Avaliações: {avaliacoes}")
print(f"Estrelas: {estrelas}")
print(f"Descrição: {descricao}")

# Parte 2: Coletar os comentários
def get_comentarios(asin, paginas=3):
    options = Options()
    # options.add_argument("--headless")
    driver = webdriver.Chrome(options=options)
    comentarios = []

    for pagina in range(1, paginas + 1):
        url = f"https://www.amazon.com.br/product-reviews/{asin}/?pageNumber={pagina}"
        driver.get(url)
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        reviews = soup.select(".review")

        if not reviews:
            print(f"Nenhum comentário encontrado na página {pagina}")
            break

        for r in reviews:
            nota = r.select_one("i.review-rating span")
            texto = r.select_one("span.review-text-content span")
            comentarios.append({
                "nota": nota.get_text(strip=True) if nota else "Nota não encontrada",
                "comentario": texto.get_text(strip=True) if texto else "Comentário vazio"
            })

    driver.quit()
    return comentarios

# Executa a coleta de comentários
asin = "B0B9PSBNYL"
comentarios = get_comentarios(asin)

# Exibe no console
for i, c in enumerate(comentarios, 1):
    print(f"\nComentário {i}:")
    print(f"Nota: {c['nota']}")
    print(f"Comentário: {c['comentario']}")

# Salva os comentários em Excel
df = pd.DataFrame(comentarios)
df.to_excel("comentarios_amazon.xlsx", index=False)
print("\n✅ Comentários salvos em 'comentarios_amazon.xlsx'")


KeyboardInterrupt: 

Código com playwright